# Dynamic Autoscaling: How Many Hosts Does SURF Need?

*Use case 1 of 3, about 10 minutes.*

We twin the SURF-SARA production cluster at SURF, the Dutch national supercomputing center.
SURF-SARA contains 277 hosts, each with 128 GB of RAM and 16 processing cores running at maximum 2.1 GHz.
OpenDT replays one week of its real-world operation (October 2022) through the OpenDC simulator.

![Datacenter digital twinning](resources/images/digital_twinning.png)

*High-level overview of datacenter digital twinning; a human-in-the-loop (HITL) oversees the process (OpenDT paper,
Figure 1).*

Run the cells in order (Shift+Enter). Before a cell that answers a question, write down a guess; the answers are folded
under *Answer*.

In [ ]:
import pandas as pd
from resources import *

check_setup()

## 1. Validating OpenDT against SURF

The first day of the trace (6-7 October 2022): predictions vs. ground truth (grey), i.e., the power draw measured at SURF.
The lower the MAPE (Mean Absolute Percentage Error), the more accurate the prediction.

In [ ]:
DAY = TRACE_START + pd.Timedelta("24h")  # the first 24 hours of the trace

day1 = simulate(until=DAY, label="OpenDT")
plot_power(day1)
print(f"MAPE: {mape(day1):.1f}%")

*Question:* with 10% as the community-accepted error rate, is OpenDT accurate enough for what-if analysis on SURF, and
in which hours is it least accurate?

<details><summary>Answer</summary>

Yes: MAPE about 6%.
OpenDT under-estimates the ground truth in almost every minute of the day, more in the first 6 hours (MAPE 10.4%)
than after (MAPE 4.6%); the OpenDT paper identifies the same underestimation trend.
</details>

## 2. What-If Analysis: Fewer and More Hosts

The same day with 1/4, 1/2, 1x, and 10x the hosts (69, 138, 277, and 2,770 hosts).

*Question:* SURF draws about 22 kW with its 277 hosts; how much would it draw with 2,770 hosts?

In [ ]:
sizes = [69, 138, 277, 2770]
runs = simulate_many("hosts", sizes, until=DAY)

table = pd.DataFrame([run.summary() for run in runs.values()], index=list(runs))
table[["avg_power_kw", "jobs_waited", "mean_wait_min", "waiting_at_end"]].round(1)

*Columns:* `avg_power_kw`, the average power draw [kW]; `jobs_waited`, the jobs that cannot start right away
(all hosts busy); `mean_wait_min`, the average waiting time per job [minutes]; `waiting_at_end`, the jobs still
waiting at the end of the day.

*Question:* how far is the power draw with 2,770 hosts from the guess, and where does the difference come from?

<details><summary>Answer</summary>

About 4x (84 vs. 22 kW): the jobs are the same, so the 2,493 extra hosts are idle, and an idle host still
draws 25 W (2,493 x 25 W = 62 kW).
Over-provisioning could lead to sustainability concerns, both environmental and financial, where energy (and thus,
money) is wasted by idle infrastructure.
</details>

*Question:* with 138 and with 69 hosts, how much power does SURF save, and what happens to the jobs?

<details><summary>Answer</summary>

With 138 hosts, only 16% less power (switching off 139 idle hosts saves 139 x 25 W, about 3.5 kW), and some
jobs wait: at peak, more than 138 jobs need a host. With 69 hosts, jobs wait more than a day on average, and hundreds
still wait at the end of the day; the power is lower because these jobs have not run yet.
Under-provisioning ICT infrastructure could lead to performance concerns, system faults, and, ultimately, system
failures.
</details>

In [ ]:
plot_power(*runs.values());  # optional: the four simulations in one plot

## 3. Hosts Needed over the Day

In the simulation, each running job occupies one whole host (16 cores): hosts in use = running jobs, every 150 s.
A dynamic autoscaler keeps these hosts on, plus a few spare ones, and switches off the rest.

*Question:* how many of the 277 hosts does SURF need over the day, at the busiest and at the quietest moment?

In [ ]:
HOSTS = 277  # hosts at SURF
in_use = day1.service["tasks_active"]  # busy hosts at each moment (one job per host)

ax = in_use.plot(label="Hosts in Use")
ax.fill_between(in_use.index, in_use, HOSTS, alpha=0.2, label="Idle Hosts")
ax.set(ylim=(0, 300), xlabel="Time [UTC]", ylabel="Hosts")
legend_above(ax)

In [ ]:
SPARE = 10  # hosts kept on for new jobs
IDLE_POWER = 25  # watts an idle SURF host draws

needed = (in_use + SPARE).clip(upper=HOSTS)
switched_off = HOSTS - needed
saved_kwh = (switched_off * IDLE_POWER).mean() / 1000 * 24  # average power [kW] x 24 hours
used_kwh = day1.summary()["energy_kwh"]
print(f"Switching off idle hosts saves {saved_kwh:.0f} of {used_kwh:.0f} kWh ({saved_kwh / used_kwh:.0%}).")

*Question:* which costs of switching hosts off and on again does this estimate leave out?

<details><summary>Answer</summary>

Boot time: hosts need minutes to boot, so new jobs wait when no spare host is free.
A human-in-the-loop, aiming to ensure the correct and ethical operation of the process, oversees at least the major
decisions (e.g., switching off hosts) and can intervene at any time.
</details>

*Exercise 1:* with `SPARE = 0` and then `SPARE = 50` in the cell above, how much energy does SURF save, and what could
go wrong without spare hosts?

<details><summary>Answer</summary>

About 18% with 0 spare hosts and about 12% with 50; with 0, every new job waits for a host to boot.
</details>

*Exercise 2 (optional):* the next cell simulates the same day with hosts that draw 10 W when idle (instead of 25 W);
how do the power draw and the MAPE change?

In [ ]:
low_idle = simulate(until=DAY, idle_power=10, label="OpenDT, 10 W idle")
plot_power(day1, low_idle)
print(f"Average power draw: {day1.summary()['avg_power_kw']:.1f} vs. {low_idle.summary()['avg_power_kw']:.1f} kW")
print(f"MAPE: {mape(day1):.1f}% vs. {mape(low_idle):.1f}%")

<details><summary>Answer</summary>

21.7 vs. 19.0 kW (12% less), but the MAPE goes to about 18%: with other hardware, the ground truth from SURF
no longer applies.
</details>

## Bonus: Changing the Hosts in the Live Demo

*Optional.* In the live demo, we change the hosts from the REST API (`PUT /api/topology`); from then on, OpenDT twins
the datacenter with 2,770 hosts. The same view, from two simulations:

In [ ]:
SWITCH = TRACE_START + pd.Timedelta("10h")  # 08:00 UTC on 7 October

scaled = day1.power["power_draw"].copy()
scaled[SWITCH:] = runs[2770].power["power_draw"][SWITCH:]  # from 08:00, the simulation with 2,770 hosts
plot_power(scaled.rename("OpenDT, 2,770 hosts from 08:00"));

*Next:* [2.calibration.ipynb](2.calibration.ipynb), simulation calibration.